# VELALAR COLLEGE OF ENGINEERING AND TECHNOLOGY, ERODE - 12
## DEPARTMENT OF INFORMATION TECHNOLOGY
### 25ITC31 - FOUNDATIONS OF DATA SCIENCE: PROJECT ASSESSMENT
---
**Student Name:** Kanishka K  
**Register Number:** 732925ITR045  
**Class:** 25IT3A  
**Scenario 14:** Electricity Consumption Management and Energy Usage Analysis System  
---
### Objective:
This notebook retrieves electricity consumption data, conducts data preprocessing (missing value handling, data type conversions, outlier detection using IQR), computes descriptive statistics, and generates comprehensive exploratory data visualizations using NumPy, Pandas, Matplotlib, and Seaborn.

## Unit 2 & 3: Environment Setup and Data Ingestion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'Segoe UI', 'DejaVu Sans', 'Arial'

# Load dataset
df = pd.read_csv('data/electricity_data.csv')
print("Dataset Shape:", df.shape)
df.head()

## Unit 4: Data Quality, Cleaning & Transformation
- Check for missing values and duplicates
- Type casting for date and categorical columns
- Outlier detection using the Interquartile Range (IQR) technique

In [ ]:
# 1. Missing Values & Duplicates Check
print("Missing Values:")
print(df.isnull().sum())

duplicates = df.duplicated(subset=['consumer_id', 'billing_month']).sum()
print(f"\nDuplicate Records: {duplicates}")

# 2. Type conversions
df['reading_date'] = pd.to_datetime(df['reading_date'])
df['connection_type'] = df['connection_type'].astype('category')
df['payment_status'] = df['payment_status'].astype('category')

# 3. Outlier identification using IQR (Unit 4)
q1 = df['units_consumed'].quantile(0.25)
q3 = df['units_consumed'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df['is_outlier'] = (df['units_consumed'] < lower_bound) | (df['units_consumed'] > upper_bound)
print(f"Q1: {q1:.2f} | Q3: {q3:.2f} | Upper Bound: {upper_bound:.2f}")
print(f"Identified Outliers (Heavy Industrial Loads): {df['is_outlier'].sum()}")

## Unit 1: Descriptive Statistics and Summary Analysis

In [ ]:
summary = df[['units_consumed', 'total_bill_amount', 'sanctioned_load_kw']].describe().T
summary['median'] = df[['units_consumed', 'total_bill_amount', 'sanctioned_load_kw']].median()
summary['skewness'] = df[['units_consumed', 'total_bill_amount', 'sanctioned_load_kw']].skew()
summary[['mean', 'median', 'std', 'min', 'max', 'skewness']].round(2)

## Unit 5: Exploratory Data Analysis & Visualizations
### 1. Monthly Consumption Trend Analysis

In [ ]:
plt.figure(figsize=(10, 5))
monthly = df.groupby(['billing_month', 'connection_type'], observed=True)['units_consumed'].sum().reset_index()
sns.lineplot(data=monthly, x='billing_month', y='units_consumed', hue='connection_type', marker='o', linewidth=2.5)
plt.title('Monthly Electricity Consumption Trend across Categories', fontweight='bold')
plt.ylabel('Energy (kWh)')
plt.show()

### 2. High-Consumption Consumer Identification (Scenario 14 Requirement 5)

In [ ]:
plt.figure(figsize=(10, 5))
top10 = df.groupby(['name', 'connection_type'], observed=True)['units_consumed'].mean().reset_index()
top10 = top10.sort_values(by='units_consumed', ascending=True).tail(10)
plt.barh(top10['name'], top10['units_consumed'], color='#dc2626')
plt.xlabel('Average Monthly Consumption (kWh)')
plt.title('Top 10 High Consumption Consumers Alert List', fontweight='bold')
plt.show()

### 3. Consumption Distribution & Outlier Detection

In [ ]:
fig, (ax_box, ax_hist) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, gridspec_kw={'height_ratios': [0.25, 0.75]})
sns.boxplot(x=df['units_consumed'], ax=ax_box, color='#38bdf8')
sns.histplot(df['units_consumed'], kde=True, ax=ax_hist, color='#2563eb')
plt.show()

### 4. Correlation Matrix Heatmap

In [ ]:
plt.figure(figsize=(7, 5))
corr = df[['sanctioned_load_kw', 'units_consumed', 'total_bill_amount']].corr()
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.3f')
plt.title('Feature Correlation Matrix', fontweight='bold')
plt.show()